In [ ]:
"""
2-phase Ensemble 
"""
#paths:
from pathlib import Path
#handling data:
import numpy as np
import pandas as pd
import xarray as xr
from yaml import load
from yaml import CLoader as Loader
from datetime import datetime
#abil functions:
from abil.tune import ModelTuner as tune
from abil.predict import ModelPredictor as predict
from abil.post import AbilPostProcessor as post
from abil.utils import example_data 
#plotting:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.colors import Normalize

In [ ]:
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in [start, *start.parents]:
        if (path / "environment.yml").exists():
            return path
    raise FileNotFoundError("Could not find project root containing environment.yml")


PROJECT_ROOT = find_project_root()
conffile = PROJECT_ROOT / "2-phase example" / "2-phase.yml"

# Load model configuration
with conffile.open('r') as f:
    model_config = load(f, Loader=Loader)

model_config['root'] = str(PROJECT_ROOT) + '/'
model_config['local_root'] = str(PROJECT_ROOT) + '/'


In [ ]:
# Load training data
targets = pd.read_csv(PROJECT_ROOT / model_config['targets'])
d = pd.read_csv(PROJECT_ROOT / model_config['training'])

# Define target
target = targets['Target'][0]

# Define predictors based on YAML
predictors = model_config['predictors']
d = d.dropna(subset=predictors)

# Split training data into X_train and y
y = d[target]
X_train = d[predictors]

print("finished loading data")


In [ ]:

#setup model:
m = tune(X_train, y, model_config)

#run model:
m.train(model="rf")
m.train(model="xgb")
m.train(model="knn")


In [ ]:
# Load prediction data
X_predict = pd.read_csv(PROJECT_ROOT / model_config['prediction'])
X_predict.set_index(["time", "depth", "lat", "lon"], inplace=True)
X_predict = X_predict.dropna(subset=predictors)
X_predict = X_predict[predictors]

# Setup model
m = predict(X_train, y, X_predict, model_config)

# Predict model
m.make_prediction()


In [ ]:

# Posts
targets = np.array([target])
current_date = datetime.today().strftime('%Y-%m-%d')

def do_post(statistic):
    #Abundance output
    m = post(X_train, y, X_predict, model_config, statistic, datatype="abundance")
    m.export_ds(current_date)

    #PIC output
    m = post(X_train, y, X_predict, model_config, statistic, datatype="pic")
    m.estimate_carbon("pg pic")
    vol_conversion = 1e3 # to convert from pg C L-1 to pg C m-3
    magnitude_conversion = 1e-24 # convert from pg C to Tg C
    integ = m.integration(m, vol_conversion=vol_conversion,
                          magnitude_conversion=magnitude_conversion)
    integ.integrated_totals(targets)
    m.export_ds(current_date)

    #POC output
    m = post(X_train, y, X_predict, model_config, statistic, datatype="poc")
    m.estimate_carbon("pg poc")
    vol_conversion = 1e3 # to convert from pg C L-1 to pg C m-3
    magnitude_conversion = 1e-24 # convert from pg C to Tg C
    integ = m.integration(m, vol_conversion=vol_conversion,
                          magnitude_conversion=magnitude_conversion)
    integ.integrated_totals(targets)
    m.export_ds(current_date)

    vol_conversion = 1e3 # to convert from pg C L-1 to pg C m-3
    magnitude_conversion = 1e-24 # convert from pg C to Tg C
    integ = m.integration(m, vol_conversion=vol_conversion,
                          magnitude_conversion=magnitude_conversion)
    integ.integrated_totals(targets)



do_post(statistic="mean")
do_post(statistic="ci95_UL")
do_post(statistic="ci95_LL")